# 03 · Person B — DR grade classifier (EfficientNet-B0)
Needs from Person A: `data/processed_aptos.zip`, `data/aptos/train.csv`, `data/processed_idrid/`, `data/idrid/labels.csv`.
Runtime ▸ Change runtime type ▸ **T4 GPU**. Total time ≈ 15 min. Output: `models/dr_classifier.pth` + metrics + figures.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, glob, json, shutil, zipfile
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt

PROJECT_PATH = '/content/drive/MyDrive/DR_Project'
MODELS_DIR = f'{PROJECT_PATH}/models'
sys.path.append(f'{PROJECT_PATH}/app')          # shared code lives in app/dr_core.py
from dr_core import *

for f in ['data/aptos', 'data/idrid', 'data/processed', 'data/processed_idrid', 'data/raw_upload',
          'models', 'outputs/gradcam_samples', 'app/samples', 'notebooks']:
    os.makedirs(f'{PROJECT_PATH}/{f}', exist_ok=True)
RAW = f'{PROJECT_PATH}/data/raw_upload'
print('raw_upload contains:', os.listdir(RAW))
import torch, torch.nn as nn
import torchvision.transforms.v2 as T
from sklearn.model_selection import train_test_split
from sklearn.metrics import cohen_kappa_score, confusion_matrix, classification_report, accuracy_score
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', dev)

## 1 · Load the preprocessed images into RAM (fast training)

In [ ]:
shutil.copy(f'{PROJECT_PATH}/data/processed_aptos.zip', '/content/aptos224.zip')
with zipfile.ZipFile('/content/aptos224.zip') as z: z.extractall('/content/aptos224')

df = pd.read_csv(f'{PROJECT_PATH}/data/aptos/train.csv')
X = np.stack([cv2.cvtColor(cv2.imread(f'/content/aptos224/{i}.png'), cv2.COLOR_BGR2RGB) for i in df['id_code']])
y = df['diagnosis'].values
print(X.shape, np.bincount(y))

## 2 · Split, augmentation, class-balanced sampling

In [ ]:
tr_idx, va_idx = train_test_split(np.arange(len(y)), test_size=0.2, stratify=y, random_state=42)
aug = T.Compose([T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(30), T.ColorJitter(0.2, 0.2)])
mean_t, std_t = torch.tensor(MEAN).view(3, 1, 1), torch.tensor(STD).view(3, 1, 1)

class DS(torch.utils.data.Dataset):
    def __init__(self, X, y, train): self.X, self.y, self.train = X, y, train
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        x = torch.from_numpy(self.X[i]).permute(2, 0, 1)             # uint8 CHW
        if self.train: x = aug(x)
        return (x.float() / 255 - mean_t) / std_t, int(self.y[i])

counts = np.bincount(y[tr_idx]); w = 1.0 / np.sqrt(counts[y[tr_idx]])      # sqrt-balanced: helps rare grades, avoids over-fitting them
sampler = torch.utils.data.WeightedRandomSampler(w, num_samples=len(tr_idx), replacement=True)
dl_tr = torch.utils.data.DataLoader(DS(X[tr_idx], y[tr_idx], True), batch_size=32, sampler=sampler, num_workers=2)
dl_va = torch.utils.data.DataLoader(DS(X[va_idx], y[va_idx], False), batch_size=64, num_workers=2)
print(len(tr_idx), 'train /', len(va_idx), 'val')

## 3 · Train (saves the best epoch by quadratic weighted kappa — the standard APTOS metric)

In [ ]:
EPOCHS = 20
model = build_classifier(pretrained=True).to(dev)
opt = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=1e-3, total_steps=EPOCHS * len(dl_tr))
crit = nn.CrossEntropyLoss(label_smoothing=0.05)
use_amp = dev == 'cuda'
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

@torch.no_grad()
def predict(model, dl):
    model.eval(); P = []
    for x, _ in dl:
        with torch.amp.autocast('cuda', enabled=use_amp): P.append(model(x.to(dev)).float().argmax(1).cpu().numpy())
    return np.concatenate(P)

best, hist = -1, []
for ep in range(1, EPOCHS + 1):
    model.train(); tot = 0
    for x, t in dl_tr:
        x, t = x.to(dev), t.to(dev)
        with torch.amp.autocast('cuda', enabled=use_amp): loss = crit(model(x).float(), t)
        opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step(); tot += loss.item()
    p = predict(model, dl_va)
    acc, qwk = accuracy_score(y[va_idx], p), cohen_kappa_score(y[va_idx], p, weights='quadratic')
    hist.append((tot / len(dl_tr), acc, qwk))
    print(f'epoch {ep:2d}  loss {tot / len(dl_tr):.3f}  val acc {acc:.3f}  val QWK {qwk:.3f}')
    if qwk > best: best = qwk; save_half(model, f'{MODELS_DIR}/{CLS_FILE}'); print('   saved best')

h = np.array(hist); fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(h[:, 0]); ax[0].set_title('Training loss'); ax[1].plot(h[:, 1], label='accuracy'); ax[1].plot(h[:, 2], label='QWK'); ax[1].legend(); ax[1].set_title('Validation')
plt.savefig(f'{PROJECT_PATH}/outputs/training_curves.png', bbox_inches='tight'); plt.show()

## 4 · Final evaluation on the APTOS validation set

In [ ]:
model = load_weights(build_classifier(), f'{MODELS_DIR}/{CLS_FILE}').to(dev)
p = predict(model, dl_va); yt = y[va_idx]
print(classification_report(yt, p, target_names=GRADES, digits=3))
cm = confusion_matrix(yt, p)
fig, ax = plt.subplots(figsize=(6, 5)); ax.imshow(cm, cmap='Blues')
for i in range(5):
    for j in range(5): ax.text(j, i, cm[i, j], ha='center', va='center', color='white' if cm[i, j] > cm.max() / 2 else 'black')
ax.set_xticks(range(5)); ax.set_yticks(range(5)); ax.set_xticklabels(GRADES, rotation=45); ax.set_yticklabels(GRADES)
ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title('APTOS validation - confusion matrix')
plt.savefig(f'{PROJECT_PATH}/outputs/confusion_matrix.png', bbox_inches='tight'); plt.show()
update_metrics(MODELS_DIR, 'classifier_aptos_validation', {'accuracy': round(accuracy_score(yt, p), 3),
    'quadratic_weighted_kappa': round(cohen_kappa_score(yt, p, weights='quadratic'), 3), 'val_images': len(yt)})

## 5 · External test on IDRiD (does it generalise to another population / camera?)

In [ ]:
idr = pd.read_csv(f'{PROJECT_PATH}/data/idrid/labels.csv')
Xi = np.stack([cv2.cvtColor(cv2.imread(f'{PROJECT_PATH}/data/processed_idrid/{i}.png'), cv2.COLOR_BGR2RGB) for i in idr['id_code']])
yi = idr['diagnosis'].values
dl_i = torch.utils.data.DataLoader(DS(Xi, yi, False), batch_size=64)
pi = predict(model, dl_i)
res = {'accuracy': round(accuracy_score(yi, pi), 3), 'quadratic_weighted_kappa': round(cohen_kappa_score(yi, pi, weights='quadratic'), 3), 'images': len(yi)}
update_metrics(MODELS_DIR, 'classifier_idrid_external', res); print(res)
print(confusion_matrix(yi, pi))

> A drop on IDRiD is **expected** (different camera, labelling and class mix = domain shift). Report it honestly — it is a finding, not a failure.

## 6 · Grad-CAM samples for the report

In [ ]:
cpu_model = load_weights(build_classifier(), f'{MODELS_DIR}/{CLS_FILE}')          # Grad-CAM runs on CPU
gc = GradCAM(cpu_model, cpu_model.features[-1])
fig, ax = plt.subplots(2, 5, figsize=(18, 7))
for g in range(5):
    ids = [i for i in va_idx if y[i] == g][:1]
    for i in ids:
        cam, probs, pred = gc(to_tensor(X[i]))
        ax[0, g].imshow(X[i]); ax[0, g].set_title(f'true {g} - {GRADES[g]}'); ax[1, g].imshow(cam_overlay(X[i], cam)); ax[1, g].set_title(f'pred {pred} ({probs[pred]:.0%})')
    ax[0, g].axis('off'); ax[1, g].axis('off')
plt.savefig(f'{PROJECT_PATH}/outputs/gradcam_samples/grid.png', bbox_inches='tight'); plt.show()

### ✅ Done → tell the team
> "`models/dr_classifier.pth` is ready, metrics are in `models/metrics.json`, figures in `outputs/`."

**If the score is low** (QWK < 0.8) try, in order: `EPOCHS = 30` · retrain on larger images (set `CLS_SIZE = 300` in `dr_core.py` — Person A must reprocess) · lower `max_lr` to `5e-4`. Re-run only cells 3-4; the best model is overwritten only if it improves within a run, so copy the old `.pth` first if you want to keep it.